# Data Analysis & Advanced Machine Learning
**Idea:** understand your data and handle it well, and the model gets better.

We train the **same XGBoost model twice**:
1. On the crude data (Chapter 2)
2. After we clean, explore and improve the data (Chapter 6)

Then we compare.

---
## Chapter 0 · Getting Started
**Example:** predict which cash-on-delivery orders will be returned (a minority).

### Your first code cell
Hover under this cell and click **+ Code**.
Type `print("Hello")`, then press **Shift + Enter**.

### Your first text cell
Click **+ Text**. Write a title with `#` and some **bold** text with `**`.

### Install the libraries
Run this cell once.

In [ ]:
%pip install -q -r requirements.txt

### Import pandas
**pandas** is our tool for tables. We call it `pd`.

In [ ]:
import pandas as pd

### Load the data
Name the table `df`.

In [ ]:
df = pd.read_csv("data/orders_raw.csv")

### First rows
1 row = 1 order. 1 column = 1 piece of information.

### Size
(rows, columns)

### Column info

### Summary numbers

### Count values
`returned`: 1 = returned, 0 = kept.

### One order
All lines = **features**. Last line = **target** (what we predict).

In [ ]:
df.iloc[0]

Our problem is **classification**.

---
## Chapter 1 · Models Speak Numbers

### Just try
Import **XGBoost**: our model library.

**Expect:** an error. XGBoost can't read text.

In [ ]:
from xgboost import XGBClassifier

try:
    XGBClassifier().fit(df.drop(columns=["returned"]), df["returned"])
except Exception as error:
    print("XGBoost says:", str(error)[:160], "...")

### Column types

In [ ]:
df.dtypes

### Empty cells
XGBoost can handle them.

In [ ]:
df.isna().sum()

### Drop what we can't use yet
IDs, dates (Chapter 5) and `price` (Chapter 3).

In [ ]:
df_raw = df.drop(columns=["order_id", "customer_id",
                         "order_datetime", "delivery_date",
                         "price"])
df_raw.head(3)

### One-hot encoding
One text column → one 0/1 column per value.

In [ ]:
example = df_raw["phone_confirmed"].head(5)
pd.concat([example, pd.get_dummies(example, prefix="phone_confirmed", dtype=int)], axis=1)

### One-hot on all text columns
Each typo gets its own column (we fix that in Chapter 3).

In [ ]:
pd.get_dummies(df_raw, columns= ["wilaya", "delivery_company", "product_category", "phone_confirmed"], dtype=int)

In [ ]:
df_raw = pd.get_dummies(df_raw, columns= ["wilaya", "delivery_company", "product_category", "phone_confirmed"], dtype=int)
df_raw.shape

Look at the table.

**Expect:** `...` hides some columns.

In [ ]:
df_raw.head()

Show all columns.

In [ ]:
pd.set_option("display.max_columns", None)
df_raw.head()

### X and y
**X** = features. **y** = target.

In [ ]:
X = df_raw.drop(columns=["returned"])
y = df_raw["returned"]
print("X:", X.shape, " y:", y.shape)

---
## Chapter 2 · The Baseline
*How good is our first model?*

### Train / test split
Import the splitter from **scikit-learn**. We hide some orders as a **test set**.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

### Train the first model

In [ ]:
from xgboost import XGBClassifier

first_model = XGBClassifier(random_state=42)
first_model.fit(X_train, y_train)


### Predict

In [ ]:
y_pred = first_model.predict(X_test)
y_pred[:20]

### Recall
**Recall:** of all the orders really returned, how many did the model catch?

In [ ]:
from sklearn.metrics import recall_score

first_recall = round(recall_score(y_test, y_pred), 2)
first_recall

**Remember this score.** We compare in Chapter 6.

---
## Chapter 3 · Clean the Mess
*Is the data right?*

### `info()`

In [ ]:
df.info()

### `describe()`

In [ ]:
df.describe().round(1)

### Wilaya spellings

In [ ]:
df["wilaya"].value_counts()

### Duplicates: count

In [ ]:
df.duplicated().sum()

### Duplicates: remove
The same order in train and test = **data leakage**.

In [ ]:
df = df.drop_duplicates()
df.shape

### Typos: lowercase + strip

In [ ]:
df["wilaya"] = df["wilaya"].str.lower()

df["wilaya"] = df["wilaya"].str.strip()

df["wilaya"].nunique()

### Typos: fix list

In [ ]:
WILAYA_FIXES = {
    "algiers": "alger", "setif": "sétif", "bejaia": "béjaïa", "tizi-ouzou": "tizi ouzou",
    "sidi bel abbes": "sidi bel abbès", "sba": "sidi bel abbès", "ghardaia": "ghardaïa",
    "ain temouchent": "aïn témouchent", "bechar": "béchar",
}
df["wilaya"] = df["wilaya"].replace(WILAYA_FIXES)
df["wilaya"].nunique()

### Category typos

In [ ]:
CATEGORY_FIXES = {"clothes": "clothing", "electronic": "electronics",
                  "accessories": "phone accessories", "sport": "sports"}

df["product_category"] = df["product_category"].str.lower()

df["product_category"] = df["product_category"].str.strip()

df["product_category"] = df["product_category"].replace(CATEGORY_FIXES)

df["product_category"].value_counts()

### Price: text → number

In [ ]:
df["price"] = df["price"].str.replace("DA", "")

df["price"] = df["price"].str.replace(" ", "")

df["price"] = df ["price"].astype(int)

### Dates: text → date

In [ ]:
df["order_datetime"] = pd.to_datetime(df["order_datetime"], format="mixed")

df["delivery_date"] = pd.to_datetime(df["delivery_date"])

df[["order_datetime", "delivery_date"]].dtypes

### Impossible values → empty
Impossible quantities and ages become empty cells.

In [ ]:
bad_quantity = df["quantity"] <= 0

bad_age = (df["customer_age"] < 15) | (df["customer_age"] > 90)

print("impossible quantities:", bad_quantity.sum(), "| impossible ages:", bad_age.sum())

df["quantity"] = df["quantity"].mask(bad_quantity)
df["customer_age"] = df["customer_age"].mask(bad_age)

### Missing values
| Case | Action |
| --- | --- |
| Mostly empty | drop the **column** |
| Very few rows | drop the **rows** |
| Useful column | **impute** (fill) |

In [ ]:
(df.isna().mean() * 100).round(1)

### Drop the column
`customer_rating`

In [ ]:
df = df.drop(columns=["customer_rating"])
df.shape

### Drop the rows
Orders with no wilaya.

In [ ]:
len(df)

In [ ]:
df = df.dropna(subset=["wilaya"])

### Impute with the median
The median = the middle value.

In [ ]:
df["delivery_fee"] = df["delivery_fee"].fillna(df["delivery_fee"].median())

df["customer_age"] = df["customer_age"].fillna(df["customer_age"].median())

### Impute with the most common value

In [ ]:
df["quantity"] = df["quantity"].fillna(df["quantity"].mode()[0])

### TODO
Check that `delivery_fee` has no empty cells.

Hint: `isna()` then `.sum()`. Expect: 0

In [ ]:
# df["delivery_fee"].____().sum()

### Save

In [ ]:
df.to_csv("data/orders_clean.csv", index=False)

print("Saved:", df.shape, "| empty cells left:", df.isna().sum().sum())

---
## Chapter 4 · Explore
*What does the data say?*

We only look. Nothing is saved.

### Checkpoint

In [ ]:
df = pd.read_csv("data/orders_clean.csv")

df["order_datetime"] = pd.to_datetime(df["order_datetime"])

df["delivery_date"] = pd.to_datetime(df["delivery_date"])

print("Checkpoint OK:", df.shape)

### Base rate

In [ ]:
base_rate = df["returned"].mean()

print(f"{base_rate:.1%} of orders are returned")

### Distribution

In [ ]:
df["customer_age"].plot.hist(bins=30, title="Customer age");

### By wilaya
Dashed line = base rate.

In [ ]:
by_wilaya = df.groupby("wilaya")["returned"].mean().sort_values()

ax = by_wilaya.plot.barh(figsize=(12, 8), title="Return rate by wilaya")

ax.axvline(base_rate, color="gray", linestyle="--");

### By category

In [ ]:
df.groupby("product_category")["returned"].mean()

In [ ]:
by_category = df.groupby("product_category")["returned"].mean().sort_values()

ax = by_category.plot.barh(title="Return rate by category")

ax.axvline(base_rate, color="gray", linestyle="--");

### Phone confirmation

In [ ]:
df.groupby("phone_confirmed")["returned"].mean().round(2)

### By hour
→ **Discovery 1**

In [ ]:
hour = df["order_datetime"].dt.hour

ax = df.groupby(hour)["returned"].mean().plot.bar(figsize=(9, 3), title="Return rate by hour")

ax.axhline(base_rate, color="gray", linestyle="--");

### By delivery days
→ **Discovery 2**

In [ ]:
delay = (df["delivery_date"] - df["order_datetime"].dt.normalize()).dt.days

ax = df.groupby(delay)["returned"].mean().plot.bar(title="Return rate by delivery days")

ax.axhline(base_rate, color="gray", linestyle="--");

### Our discoveries
| # | Discovery | Column for Chapter 5 |
| --- | --- | --- |
| 1 | Night orders | `order_hour` |
| 2 | Long deliveries | `delivery_days` |

---
## Chapter 5 · Engineer the Signal
*Turn discoveries into columns.*

### Discovery 1 → `order_hour`

In [ ]:
df["order_hour"] = df["order_datetime"].dt.hour
df[["order_datetime", "order_hour"]].head(3)

### Discovery 2 → `delivery_days`

In [ ]:
df["delivery_days"] = (df["delivery_date"] - df["order_datetime"].dt.normalize()).dt.days
df[["order_datetime", "delivery_date", "delivery_days"]].head(3)

### What we do NOT add
"Customer return rate" uses the answer (`returned`). This is **target leakage**.

A feature may only use what we knew **when the order was placed**.

### Final table

In [ ]:
final = df.drop(columns=["order_id", "customer_id", "order_datetime", "delivery_date"])

In [ ]:
final = pd.get_dummies(final, dtype=int)
final.shape

In [ ]:
final.head()

### Save

In [ ]:
final.to_csv("data/orders_features.csv", index=False)

---
## Chapter 6 · The Final Model
*Same model, better data. Does it improve?*

In [ ]:
from sklearn.model_selection import train_test_split

X = final.drop(columns=["returned"])
y = final["returned"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train.shape

### Class imbalance

In [ ]:
y_train.value_counts()

### Class weight
Tell XGBoost that a return counts more: **kept orders ÷ returned orders**.

In [ ]:
weight = (y_train == 0).sum() / (y_train == 1).sum()
weight

### Train the final model
Same XGBoost. We choose some **hyperparameters** and the class weight.

In [ ]:
from xgboost import XGBClassifier

final_model = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, scale_pos_weight=weight, random_state=42)
final_model.fit(X_train, y_train)

### Score

In [ ]:
y_pred = final_model.predict(X_test)
final_recall = round(recall_score(y_test, y_pred), 2)
final_recall

### First vs final

In [ ]:
comparison = pd.Series({"first model": first_recall, "final model": final_recall})
comparison

In [ ]:
comparison.plot.bar(rot=0, ylim=(0, 1), title="Same XGBoost, better data");

### Feature importance 

In [ ]:
importance = pd.Series(final_model.feature_importances_, index=X.columns)

importance.sort_values().plot.barh(figsize=(12,10));